# LeetCode #1155 — Number of Dice Rolls With Target Sum
**Mức độ:** Medium · **Kỹ thuật:** Dynamic Programming (Quy hoạch động), Đếm số cách, Rolling Array · **Ngôn ngữ:** Python 3.

> **Tư duy cốt lõi:** Muốn biết có bao nhiêu cách gieo `i` viên xúc xắc được tổng `s`, hãy thử mặt của viên **cuối cùng**, rồi cộng số cách đạt `s - mặt` bằng `i-1` viên trước đó.

Bài học đi từ cộng các kết quả nhỏ và vòng lặp Python đến bảng DP, cách nộp LeetCode, tối ưu bộ nhớ, kiểm chứng và tự luyện.

## 1. Mục tiêu học tập

Sau notebook, bạn sẽ:
1. Hiểu bài toán **đếm số cách**, phân biệt với việc tìm một cách hợp lệ.
2. Làm quen `range`, vòng lặp lồng nhau, `list`, phép chia lấy dư `%` trong Python.
3. Hiểu ý nghĩa trạng thái `dp[i][s]` và **vì sao phải bắt đầu từ `dp[0][0]=1`**.
4. Tự suy ra công thức chuyển trạng thái, lập bảng và trace từng viên xúc xắc.
5. Biết bản DP 2D `O(n×k×target)`, bản DP 1D và một cải tiến nâng cao bằng Sliding Window.

## 2. Phát biểu đề gốc và ví dụ

Có `n` viên xúc xắc; mỗi viên có `k` mặt mang số từ `1` đến `k`. Hỏi có bao nhiêu **dãy kết quả có thứ tự** khi gieo `n` viên mà tổng bằng `target`? Trả số cách theo modulo `1_000_000_007`.

| `n` | `k` | `target` | Số cách | Giải thích |
|---:|---:|---:|---:|---|
| 1 | 6 | 3 | 1 | Chỉ `(3)`. |
| 2 | 6 | 7 | 6 | `(1,6),(2,5),...,(6,1)`. |
| 30 | 30 | 500 | 222616187 | Kết quả sau `% 1_000_000_007`. |

**Ràng buộc:** `1<=n,k<=30`, `1<=target<=1000`.

> Hai dãy `(1,6)` và `(6,1)` được tính là **hai cách** vì kết quả theo từng viên xúc xắc khác nhau. Đây không phải bài chọn các số **không xét thứ tự**.

In [1]:
print('Các cách 2 xúc xắc 6 mặt đạt tổng 7:')
ways = [(first,7-first) for first in range(1,7) if 1<=7-first<=6]
print(ways)
assert len(ways) == 6

Các cách 2 xúc xắc 6 mặt đạt tổng 7:
[(1, 6), (2, 5), (3, 4), (4, 3), (5, 2), (6, 1)]


## 3. Python nền tảng: `range` và vòng lặp lồng nhau

`range(1,k+1)` tạo các số nguyên `1,2,...,k`. Điểm cuối của `range` **không được lấy**, nên viết `k+1` mới bao gồm mặt `k`.

Khi có hai xúc xắc, dùng hai vòng `for` để liệt kê mọi cặp kết quả có thứ tự.

In [2]:
k = 3
all_pairs = []
for a in range(1,k+1):
    for b in range(1,k+1):
        all_pairs.append((a,b))
print('Tất cả cặp khi k=3:',all_pairs)
print('Các cặp có tổng 4:',[pair for pair in all_pairs if sum(pair)==4])
assert len(all_pairs) == 9

Tất cả cặp khi k=3: [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
Các cặp có tổng 4: [(1, 3), (2, 2), (3, 1)]


## 4. Python nền tảng: `list` và truy cập bằng chỉ số

Ta dùng một `list` để lưu **số cách tạo ra từng tổng**. Ví dụ `ways[4]` là số cách đạt tổng 4. `ways = [0]*8` tạo danh sách 8 số 0, có chỉ số 0–7.

Câu `ways[s] += count` là viết ngắn cho `ways[s] = ways[s] + count`.

In [3]:
ways = [0]*8
ways[4] += 1
ways[4] += 2
print('Bảng đếm:',ways)
print('Số cách được lưu ở tổng 4:',ways[4])
assert ways[4] == 3

Bảng đếm: [0, 0, 0, 0, 3, 0, 0, 0]
Số cách được lưu ở tổng 4: 3


## 5. Python nền tảng: modulo `%`

Số dãy có thể rất lớn (tối đa `k**n` khả năng). Đề yêu cầu kết quả theo **modulo `1_000_000_007`** (thường viết `MOD`).

`a % MOD` là số dư khi chia `a` cho `MOD`. Nhờ quy tắc `(a+b) % MOD = ((a%MOD)+(b%MOD))%MOD`, ta có thể lấy modulo **trong quá trình cộng** mà không làm sai đáp án cuối.

In [4]:
MOD = 1_000_000_007
print('17 % 5 =',17%5)
print('(MOD+9) % MOD =',(MOD+9)%MOD)
assert (MOD+9)%MOD == 9

17 % 5 = 2
(MOD+9) % MOD = 9


## 6. Cách trực giác: thử toàn bộ kết quả `k^n`

Mỗi viên có `k` lựa chọn; `n` viên có `k^n` **dãy kết quả**. Ta có thể liệt kê mọi dãy bằng `itertools.product(..., repeat=n)`, lấy tổng và đếm.

Cách này rất dễ hiểu nhưng **không thể dùng cho ràng buộc lớn**: với `n=30`, `k=30`, có `30^30` khả năng.

In [5]:
from itertools import product

def dice_bruteforce(n,k,target):
    count = 0
    for rolls in product(range(1,k+1),repeat=n):
        if sum(rolls) == target:
            count += 1
    return count

assert dice_bruteforce(2,3,4) == 3
assert dice_bruteforce(2,6,7) == 6
print('2 xúc xắc 3 mặt, tổng 4:',dice_bruteforce(2,3,4))

2 xúc xắc 3 mặt, tổng 4: 3


## 7. Tại sao cần Dynamic Programming?

Nếu cứ liệt kê tất cả kết quả, cùng một bài toán nhỏ (ví dụ **2 viên còn lại đạt tổng 5**) sẽ được tính lại trong rất nhiều nhánh.

**Quy hoạch động** (*Dynamic Programming*, DP) lưu đáp án các bài toán nhỏ để tái sử dụng. Thay vì nhớ toàn bộ dãy gieo, ta chỉ nhớ:
- Đã gieo bao nhiêu viên xúc xắc?
- Tổng hiện cần là bao nhiêu?

Hai thông tin ấy đủ để xác định số cách, vì các viên cùng có `k` mặt độc lập.

## 8. Xác định trạng thái `dp[i][s]`

Định nghĩa:

\[dp[i][s]=\text{số cách gieo đúng }i\text{ viên để được tổng }s\]

Ví dụ với `k=3`:
- `dp[1][2] = 1`: gieo một viên được 2 chỉ bằng `(2)`.
- `dp[2][4] = 3`: `(1,3),(2,2),(3,1)`.
- `dp[2][7] = 0`: hai viên 3 mặt chỉ đạt tổng tối đa 6.

Bảng DP có `n+1` hàng (`i=0..n`) và `target+1` cột (`s=0..target`).

In [6]:
n,k,target = 2,3,4
dp = [[0]*(target+1) for _ in range(n+1)]
print('Số hàng:',len(dp),'| số cột:',len(dp[0]))
for row in dp:
    print(row)

Số hàng: 3 | số cột: 5
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]
[0, 0, 0, 0, 0]


## 9. Điều kiện gốc: vì sao `dp[0][0] = 1` chứ không phải `0`?

Khi chưa gieo viên nào (`i=0`), có đúng **một cách** để đạt tổng 0: **không gieo gì cả** (dãy rỗng).

Vì vậy:

\[dp[0][0]=1,\qquad dp[0][s]=0\quad(s>0)\]

Đây là **cách đếm**, không phải số mặt xuất hiện. Nếu gán `dp[0][0]=0`, mọi trạng thái kế tiếp cộng từ số 0 nên cuối cùng đếm được 0 ở tất cả các ô.

In [7]:
dp = [[0]*5 for _ in range(3)]
dp[0][0] = 1
print('Hàng i=0:',dp[0])
assert dp[0] == [1,0,0,0,0]

Hàng i=0: [1, 0, 0, 0, 0]


## 10. Công thức chuyển trạng thái — xét mặt của viên cuối

Muốn `i` viên đạt tổng `s`, viên cuối phải có một giá trị `face` thuộc `1..k`.

Nếu viên cuối ra `face`, `i-1` viên trước phải có tổng `s-face`. Các lựa chọn mặt cuối **không trùng nhau**, vì vậy cộng số cách:

\[\boxed{dp[i][s]=\sum_{face=1}^{k} dp[i-1][s-face]}\]

Chỉ cộng khi `s-face >= 0` (nếu âm thì không có cách).

**Ví dụ:** `k=3`, `i=2`, `s=4`:

`dp[2][4] = dp[1][3] + dp[1][2] + dp[1][1] = 1+1+1=3`.

Ta đã thu nhỏ bài toán `2 viên, tổng 4` thành ba bài toán `1 viên`.

In [8]:
previous_row = [0,1,1,1,0]  # một viên k=3
s = 4
for face in range(1,4):
    print(f'Viên cuối ra {face}: số cách trước đó = dp[1][{s-face}] = {previous_row[s-face]}')
print('Tổng:',sum(previous_row[s-face] for face in range(1,4)))
assert sum(previous_row[s-face] for face in range(1,4)) == 3

Viên cuối ra 1: số cách trước đó = dp[1][3] = 1
Viên cuối ra 2: số cách trước đó = dp[1][2] = 1
Viên cuối ra 3: số cách trước đó = dp[1][1] = 1
Tổng: 3


## 11. Code DP 2D — cách dễ học nhất

Vòng ngoài duyệt số viên `i`, vòng giữa duyệt tổng `s`, vòng trong thử giá trị mặt `face`.

`dp[i-1][s-face]` luôn thuộc **hàng trước**, nên mọi giá trị được dùng đều đã tính xong. Điều này cũng giúp tránh đếm nhầm một viên hai lần.

In [9]:
def dice_dp_2d(n,k,target):
    MOD = 1_000_000_007
    if target < n or target > n*k:
        return 0
    dp = [[0]*(target+1) for _ in range(n+1)]
    dp[0][0] = 1
    for i in range(1,n+1):
        for s in range(1,target+1):
            for face in range(1,min(k,s)+1):
                dp[i][s] = (dp[i][s] + dp[i-1][s-face]) % MOD
    return dp[n][target]

assert dice_dp_2d(2,3,4) == 3
assert dice_dp_2d(2,6,7) == 6
print('Kết quả:',dice_dp_2d(2,3,4))

Kết quả: 3


## 12. Trace bằng tay bảng DP cho `n=2, k=3, target=5`

| Số viên / tổng | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---:|---:|---:|---:|---:|---:|
| `i=0` | 1 | 0 | 0 | 0 | 0 | 0 |
| `i=1` | 0 | 1 | 1 | 1 | 0 | 0 |
| `i=2` | 0 | 0 | 1 | 2 | 3 | 2 |

Giải thích hàng `i=2`:
- Tổng 2: chỉ `(1,1)` → 1 cách.
- Tổng 3: `(1,2),(2,1)` → 2 cách.
- Tổng 4: `(1,3),(2,2),(3,1)` → 3 cách.
- Tổng 5: `(2,3),(3,2)` → 2 cách.

Các ô dưới tổng nhỏ nhất `i` bằng 0; các ô trên tổng lớn nhất `i*k` cũng bằng 0.

In [10]:
def trace_dice_dp(n,k,target):
    dp = [[0]*(target+1) for _ in range(n+1)]
    dp[0][0] = 1
    print('i=0:',dp[0])
    for i in range(1,n+1):
        for s in range(1,target+1):
            contributions = [dp[i-1][s-face] for face in range(1,min(k,s)+1)]
            dp[i][s] = sum(contributions)
        print(f'i={i}:',dp[i])
    return dp[n][target]

assert trace_dice_dp(2,3,5) == 2

i=0: [1, 0, 0, 0, 0, 0]
i=1: [0, 1, 1, 1, 0, 0]
i=2: [0, 0, 1, 2, 3, 2]


## 13. Điều kiện bất khả thi — cắt sớm

`n` viên, mỗi viên có số nhỏ nhất `1` nên **tổng nhỏ nhất là `n`**; mỗi viên tối đa `k` nên **tổng lớn nhất là `n*k`**.

Nếu `target < n` hoặc `target > n*k`, không có cách nào, trả `0` ngay mà không cần tạo bảng DP.

Ví dụ `n=3,k=2`:
- Tổng nhỏ nhất 3, lớn nhất 6.
- `target=2` hoặc `target=7` đều vô nghiệm.

In [11]:
assert dice_dp_2d(3,2,2) == 0
assert dice_dp_2d(3,2,7) == 0
assert dice_dp_2d(3,2,3) == 1  # (1,1,1)
assert dice_dp_2d(3,2,6) == 1  # (2,2,2)
print('PASS các tổng ngoài/ở ranh giới.')

PASS các tổng ngoài/ở ranh giới.


## 14. Tối ưu bộ nhớ: chỉ giữ hàng trước

Công thức `dp[i][s]` chỉ đọc từ `dp[i-1][...]`; vì vậy không cần lưu **tất cả** `n+1` hàng. Ta dùng hai list:

- `previous[s]`: số cách của `i-1` viên.
- `current[s]`: số cách của `i` viên.

Tính xong một hàng thì gán `previous = current` (đổi tên tham chiếu đến list vừa tính). Ta phải tạo **list mới cho current ở mỗi lượt**; nếu sửa trực tiếp previous khi chưa xong sẽ dùng dữ liệu của cùng một viên nhiều lần.

In [12]:
def dice_dp_1d(n,k,target):
    MOD = 1_000_000_007
    if target < n or target > n*k:
        return 0
    previous = [0]*(target+1)
    previous[0] = 1
    for i in range(1,n+1):
        current = [0]*(target+1)
        for s in range(i,min(target,i*k)+1):
            for face in range(1,min(k,s)+1):
                current[s] = (current[s] + previous[s-face]) % MOD
        previous = current
    return previous[target]

for args in [(1,6,3),(2,6,7),(2,3,4),(3,2,4),(3,2,7)]:
    assert dice_dp_1d(*args) == dice_dp_2d(*args)
print('PASS DP 1 chiều so với DP 2 chiều.')

PASS DP 1 chiều so với DP 2 chiều.


## 15. Quan sát bản 1D chuyển từng hàng

Ví dụ `n=2, k=3, target=5`. Khi `i=1`, bảng `previous` chứa số cách của một viên. Khi `i=2`, ta chỉ cần bảng cũ và bảng mới. Không cần giữ hàng `i=0` nữa.

In [13]:
previous = [1,0,0,0,0,0]
print('0 viên:',previous)
for i in range(1,3):
    current = [0]*6
    for s in range(1,6):
        for face in range(1,min(3,s)+1):
            current[s] += previous[s-face]
    print(f'{i} viên:',current)
    previous = current
assert previous == [0,0,1,2,3,2]

0 viên: [1, 0, 0, 0, 0, 0]
1 viên: [0, 1, 1, 1, 0, 0]
2 viên: [0, 0, 1, 2, 3, 2]


## 16. Code nộp LeetCode — DP 1D dễ bảo trì

Bản dưới là lời giải khuyến nghị cho người mới: tốc độ đủ với ràng buộc `n,k<=30` và `target<=1000`, bộ nhớ thấp.

Phương thức tên `numRollsToTarget`, tham số theo thứ tự `n,k,target`, trả một số nguyên đã lấy modulo.

In [14]:
class Solution:
    def numRollsToTarget(self, n: int, k: int, target: int) -> int:
        MOD = 1_000_000_007
        if target < n or target > n*k:
            return 0

        prev = [0]*(target+1)
        prev[0] = 1
        for dice in range(1,n+1):
            curr = [0]*(target+1)
            for s in range(dice,min(target,dice*k)+1):
                for face in range(1,min(k,s)+1):
                    curr[s] = (curr[s]+prev[s-face]) % MOD
            prev = curr
        return prev[target]

sol = Solution()
assert sol.numRollsToTarget(1,6,3) == 1
assert sol.numRollsToTarget(2,6,7) == 6
assert sol.numRollsToTarget(30,30,500) == 222616187
print('PASS cả 3 ví dụ của LeetCode.')

PASS cả 3 ví dụ của LeetCode.


## 17. Vì sao DP đúng? — chứng minh bằng quy nạp

**Cơ sở `i=0`:** chỉ có một dãy rỗng tổng 0, nên `prev[0]=1`; tổng dương không thể có.

**Bước quy nạp:** giả sử `prev[s]` chứa đúng số cách cho `i-1` viên. Mỗi dãy hợp lệ của `i` viên có **duy nhất** một mặt cuối `face` trong `1..k`, còn `i-1` viên đầu phải đạt `s-face`. Bởi các nhóm phân theo mặt cuối không giao nhau, số cách tổng bằng tổng các `prev[s-face]`.

Do vậy, `curr[s]` đếm **đủ và không trùng**; sau khi hoàn tất `n` viên, `prev[target]` là đáp án. Phép `% MOD` giữ đúng đáp án theo modulo.

## 18. Độ phức tạp chính xác

Gọi `n` là số viên, `k` là số mặt, `T=target`.

| Phương pháp | Thời gian | Bộ nhớ phụ |
|---|---|---|
| Vét cạn kết quả | `O(n·k^n)` | `O(n)` cho dãy đang duyệt (không lưu tất cả) |
| DP 2D | **`O(n·k·T)`** | `O(n·T)` |
| DP 1D | **`O(n·k·T)`** | **`O(T)`** |
| DP + Sliding Window nâng cao | **`O(n·T)`** | **`O(T)`** |

Giới hạn tổng khả thi trong từng hàng giúp chạy nhanh hơn thực tế, nhưng không đổi các cận trên Big-O này.

## 19. Nâng cao: giảm thời gian `O(n×k×T)` xuống `O(n×T)`

Đối với một số `s`, công thức là:

\[curr[s] = prev[s-1]+prev[s-2]+\cdots+prev[s-k]\]

Đây là **tổng một đoạn độ dài tối đa `k` trong hàng `prev`**. Khi tăng `s` lên một, đoạn cộng dịch sang phải:
- Thêm `prev[s-1]`;
- Bớt `prev[s-k-1]` nếu chỉ số đó không âm.

Gọi `window` là tổng hiện hành (theo modulo):

```python
window = (window + prev[s-1]) % MOD
if s-k-1 >= 0:
    window = (window - prev[s-k-1]) % MOD
curr[s] = window
```

Không còn vòng lặp theo `face`. Đây là kỹ thuật Sliding Window **trên hàng DP**, khác với việc trượt trên dữ liệu đầu vào của những bài mảng.

In [15]:
def dice_dp_sliding(n,k,target):
    MOD = 1_000_000_007
    if target < n or target > n*k:
        return 0
    prev = [0]*(target+1)
    prev[0] = 1
    for dice in range(1,n+1):
        curr = [0]*(target+1)
        window = 0
        for s in range(1,target+1):
            window = (window + prev[s-1]) % MOD
            if s-k-1 >= 0:
                window = (window - prev[s-k-1]) % MOD
            curr[s] = window
        prev = curr
    return prev[target]

assert dice_dp_sliding(2,3,4) == 3
assert dice_dp_sliding(30,30,500) == 222616187
print('PASS phiên bản O(n*T).')

PASS phiên bản O(n*T).


## 20. Trace `window` khi `k=3` và đã có một viên

Giả sử `prev=[0,1,1,1,0,0]` là số cách với **một viên có ba mặt**.

| `s` | Những ô `prev` cần cộng | `window` = `curr[s]` |
|---:|---|---:|
| 1 | `prev[0]` | 0 |
| 2 | `prev[1]+prev[0]` | 1 |
| 3 | `prev[2]+prev[1]+prev[0]` | 2 |
| 4 | `prev[3]+prev[2]+prev[1]` | 3 |
| 5 | `prev[4]+prev[3]+prev[2]` | 2 |

Ở `s=4`, ta **phải trừ** `prev[0]` (bằng 0); ở `s=5`, trừ `prev[1]` (bằng 1). Đây là lý do dùng `s-k-1`.

In [16]:
prev = [0,1,1,1,0,0]
window = 0
for s in range(1,6):
    add = prev[s-1]
    remove = prev[s-4] if s-4>=0 else 0
    window += add-remove
    print(f's={s}, +{add}, -{remove}, window={window}')
assert window == 2

s=1, +0, -0, window=0
s=2, +1, -0, window=1
s=3, +1, -0, window=2
s=4, +1, -0, window=3
s=5, +0, -1, window=2


## 21. Test mẫu, trường hợp biên và bài toán không thể

Cần thử:
- Một viên, một mặt;
- Tổng nhỏ nhất `n` hoặc lớn nhất `n*k`;
- Tổng bé hơn `n` hoặc vượt `n*k`;
- Nhiều viên và modulo lớn;
- Các trường hợp có nhiều dãy cùng tổng.

In [17]:
cases = [
    (1,1,1,1), (1,6,3,1), (1,6,7,0), (2,6,2,1),
    (2,6,7,6), (2,6,12,1), (2,6,13,0),
    (2,3,4,3), (3,2,3,1), (3,2,4,3),
    (3,2,5,3), (3,2,6,1), (3,2,7,0),
    (30,30,500,222616187),
]
for n,k,target,expected in cases:
    assert sol.numRollsToTarget(n,k,target) == expected,(n,k,target)
    assert dice_dp_sliding(n,k,target) == expected,(n,k,target)
print('PASS',len(cases),'test mẫu/biên, gồm ví dụ n=30.')

PASS 14 test mẫu/biên, gồm ví dụ n=30.


## 22. Kiểm thử đối chiếu độc lập với Brute Force

Với các giá trị nhỏ (`n<=4, k<=4`), việc liệt kê tất cả `k^n` kết quả còn khả thi. Ta dùng nó làm **đáp án độc lập** để so sánh ba phiên bản DP.

In [18]:
checked = 0
for n in range(1,5):
    for k in range(1,5):
        for target in range(1,n*k+3):
            expected = dice_bruteforce(n,k,target)
            two_d = dice_dp_2d(n,k,target)
            one_d = sol.numRollsToTarget(n,k,target)
            window = dice_dp_sliding(n,k,target)
            assert expected%MOD == two_d == one_d == window,(n,k,target,expected,two_d,one_d,window)
            checked += 1
print('PASS',checked,'trường hợp đối chiếu Brute Force / DP 2D / DP 1D / DP Window.')

PASS 132 trường hợp đối chiếu Brute Force / DP 2D / DP 1D / DP Window.


## 23. Kiểm tra ngẫu nhiên DP 1D và bản nâng cao

Brute Force không thể kiểm tra các đầu vào lớn. Ta có thể kiểm tra thêm vài trăm bộ tham số ở kích thước vừa, so sánh **hai cách DP có công thức triển khai khác nhau**.

In [19]:
import random
rng = random.Random(1155)
for _ in range(150):
    n = rng.randint(1,15)
    k = rng.randint(1,15)
    target = rng.randint(1,250)
    ans1 = sol.numRollsToTarget(n,k,target)
    ans2 = dice_dp_sliding(n,k,target)
    assert ans1 == ans2,(n,k,target,ans1,ans2)
print('PASS 150 test ngẫu nhiên.')

PASS 150 test ngẫu nhiên.


## 24. Bốn lỗi người mới thường gặp

**Sai 1 — Đặt `dp[0][0] = 0`.** Nếu không có “một cách tạo tổng 0 từ 0 viên”, ta sẽ không khởi tạo được bất kỳ cách nào tiếp theo.

**Sai 2 — Quên thứ tự kết quả.** Hai viên ra `(1,6)` và `(6,1)` là hai cách khác nhau, không phải một tổ hợp không thứ tự.

**Sai 3 — Sửa `prev` trực tiếp khi đang tính cùng một hàng.** Khi đó bạn có thể dùng một mặt xúc xắc nhiều lần trong cùng lượt, đếm sai. Hãy tạo `curr` mới.

**Sai 4 — Duyệt `face` tới `k` ngay cả khi `face>s`.** `prev[s-face]` sẽ có chỉ số âm; trong Python, chỉ số âm truy cập phần tử ở **cuối list**, chứ không báo lỗi! Sửa bằng `range(1,min(k,s)+1)` hoặc kiểm tra `s-face>=0`.

In [20]:
a = [10,20,30]
print('a[-1] là phần tử CUỐI, không phải vị trí -1 không tồn tại:',a[-1])
print('Vì thế phải chặn s-face < 0 trong DP.')
assert a[-1] == 30

a[-1] là phần tử CUỐI, không phải vị trí -1 không tồn tại: 30
Vì thế phải chặn s-face < 0 trong DP.


## 25. Nhận diện pattern DP đếm cách

Bạn nên nghĩ tới DP nếu:
1. Đề hỏi **bao nhiêu cách** chứ không chỉ hỏi có cách nào không.
2. Mỗi bước chọn một trong vài lựa chọn (`1..k`), và kết quả phụ thuộc vào **số bước đã làm** cùng **tổng còn lại**.
3. Nhiều nhánh đệ quy dẫn tới cùng một trạng thái (`i`, `s`).

Mẫu tổng quát: `ways[steps][value] += ways[steps-1][value-choice]`. Hãy xác định rõ liệu **thứ tự** các lựa chọn có được tính là cách khác nhau không.

In [21]:
# Biến thể mini: dùng 3 đồng xu, mỗi xu góp 0 hoặc 1, đếm số cách được tổng 2.
count = [1,0,0]
for _ in range(3):
    next_count = [0,0,0]
    for total in range(3):
        for coin in (0,1):
            if total>=coin:
                next_count[total] += count[total-coin]
    count = next_count
print('3 lần chọn 0/1 để tổng 2:',count[2])
assert count[2] == 3

3 lần chọn 0/1 để tổng 2: 3


## 26. Tự luyện — 5 bài tăng dần

**Bài 1 (tính tay).** Liệt kê tất cả cách của `n=2,k=3,target=4`, rồi viết bảng `dp[1]` và `dp[2]`.

**Bài 2 (Python).** Viết hàm kiểm tra `target` có nằm trong đoạn `[n,n*k]` hay không.

**Bài 3 (baseline).** Viết hàm thử mọi cách bằng `product` cho input nhỏ.

**Bài 4 (DP).** Viết lại DP 1D từ đầu, chỉ dùng `prev` và `curr`; nhớ `% MOD`.

**Bài 5 (biến thể).** Đếm số dãy của `n` viên xúc xắc có tổng **không vượt quá `limit`** (thay vì đúng bằng một target). Có thể cộng các trạng thái `dp[n][s]` với `s<=limit`.

In [22]:
# Các TODO không được gọi tự động; hãy tự mở comment test sau khi điền.
def ex1_list_rolls(n,k,target):
    # TODO: trả list các tuple hợp lệ
    pass

def ex2_feasible(n,k,target):
    # TODO: True khi có thể đạt tổng target
    pass

def ex3_bruteforce(n,k,target):
    # TODO: trả số cách (input nhỏ)
    pass

def ex4_dp_1d(n,k,target):
    # TODO: công thức DP dùng prev/curr
    pass

def ex5_at_most(n,k,limit):
    # TODO: số cách tổng <= limit, theo MOD
    pass

# assert len(ex1_list_rolls(2,3,4)) == 3
# assert ex2_feasible(3,2,7) is False
# assert ex3_bruteforce(2,3,4) == 3
# assert ex4_dp_1d(2,6,7) == 6
# assert ex5_at_most(2,3,4) == 6

## 27. Gợi ý 3 mức

**Mức 1 — hướng tư duy:** với `i` viên, xét **mặt của viên cuối**. Nếu tổng đích là `s`, phần trước phải tạo `s-face`.

**Mức 2 — kỹ thuật:** đặt `prev[0]=1`; với mỗi viên tạo `curr=[0]*(target+1)`; cho `face` đi từ 1 đến `min(k,s)`.

**Mức 3 — gần code:** `curr[s] = sum(prev[s-face] for face in range(1,min(k,s)+1)) % MOD`; sau mỗi viên gán `prev=curr`. Bài 5 hãy tính cả bảng đến `limit`, rồi lấy `sum(prev[:limit+1]) % MOD`.

## 28. Đáp án tham khảo — chỉ xem sau khi làm

Các lời giải sau tách riêng với bài TODO để notebook vẫn chạy sạch từ trên xuống dưới. Bạn hãy sửa TODO ở trên và dùng các test để kiểm tra trước khi đọc phần này.

In [23]:
def answer1_rolls(n,k,target):
    return [rolls for rolls in product(range(1,k+1),repeat=n) if sum(rolls)==target]

def answer2_feasible(n,k,target):
    return n <= target <= n*k

def answer3_bruteforce(n,k,target):
    return len(answer1_rolls(n,k,target))

def answer4_dp_1d(n,k,target):
    return dice_dp_1d(n,k,target)

def answer5_at_most(n,k,limit):
    MOD = 1_000_000_007
    if limit < n:
        return 0
    prev = [0]*(limit+1)
    prev[0] = 1
    for i in range(1,n+1):
        curr = [0]*(limit+1)
        for s in range(i,min(limit,i*k)+1):
            for face in range(1,min(k,s)+1):
                curr[s] = (curr[s] + prev[s-face])%MOD
        prev = curr
    return sum(prev)%MOD

assert answer1_rolls(2,3,4) == [(1,3),(2,2),(3,1)]
assert answer2_feasible(3,2,7) is False
assert answer3_bruteforce(2,3,4) == 3
assert answer4_dp_1d(2,6,7) == 6
assert answer5_at_most(2,3,4) == 6
print('PASS cả 5 đáp án luyện tập.')

PASS cả 5 đáp án luyện tập.


## 29. Tổng kết và checklist LeetCode

**Công thức nhớ nhanh:** `dp[i][s] = Σ_{face=1..k} dp[i-1][s-face]` (chỉ dùng chỉ số không âm), `dp[0][0]=1`.

**Checklist:**
- [ ] Đếm **các dãy có thứ tự**.
- [ ] Trả 0 cho tổng ngoài `[n,n*k]`.
- [ ] Mỗi hàng DP sử dụng **hàng trước**, không tự đọc hàng đang cập nhật.
- [ ] Lấy kết quả `% 1_000_000_007`.
- [ ] Signature `numRollsToTarget(self,n,k,target)` đúng.
- [ ] Bản 1D: thời gian `O(n*k*target)`, bộ nhớ `O(target)`.

**Tài liệu đối chiếu:** [LeetCode #1155](https://leetcode.com/problems/number-of-dice-rolls-with-target-sum/), [LeetCode Wiki #1155](https://leetcode.doocs.org/en/lc/1155/). Nội dung sư phạm theo cẩm nang LeetCode tiếng Việt đã cung cấp.